# Feature Extraction Pipeline

This notebook is organized into three reusable blocks:
- Utils block
- Primary features block
- Secondary features block

## Utils Block

## Feature Dictionary

### Identifiers (Join Keys, Not Model Features)

1. `review_id`: Unique review key.
2. `user_id`: Reviewer identifier.
3. `business_id`: Business identifier.
4. `date`

### Primary Features

1. `stars` (Review): Initial rating given by the user.
2. `useful` (Review): Useful-vote count on the review.
3. `funny` (Review): Funny-vote count on the review.
4. `cool` (Review): Cool-vote count on the review.
5. `review_count` (User): User total written reviews.
6. `fans` (User): User fan count.
7. `friends_count` (User): Number of friend IDs in `friends`.
8. `compliment_writer` (User): Writing-related compliment count.

### Secondary Features (With Calculations)

1. `reviewer_bias`: Difference between the current review rating and the user's own historical average rating (not a business-level average).
   Formula: $reviewer\_bias = stars - user\_average\_stars$

2. `user_tenure_days`: Time on platform when the review was written.
   Formula: $user\_tenure\_days = (review\_date - yelping\_since)$ in days.

3. `elite_persistence`: Long-term elite history.
   Calculation: number of entries in the `elite` list/string.

4. `review_word_count`: Metadata complexity by word length.
   Formula: $review\_word\_count = \text{number of whitespace-split tokens in } text$

5. `review_frequency`: Number of reviews a user made within the filtered time span used.
   Formula: $review\_frequency = \text{count of rows per } user\_id \text{ in filtered reviews}$

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable

import pandas as pd


# Default file names in the Yelp dataset folder.
REVIEW_FILE = "yelp_academic_dataset_review.json"
BUSINESS_FILE = "yelp_academic_dataset_business.json"
USER_FILE = "yelp_academic_dataset_user.json"


def read_jsonl_columns(path: str | Path, columns: Iterable[str], nrows: int | None = None) -> pd.DataFrame:
    """Read selected columns from a JSONL file to keep memory usage controlled."""
    path = Path(path)
    df = pd.read_json(path, lines=True, nrows=nrows)
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns in {path.name}: {missing}")
    return df.loc[:, list(columns)].copy()


def _friends_to_count(value: object) -> int:
    """Convert Yelp friends field into a numeric friend count."""
    if pd.isna(value) or value in (None, "", "None"):
        return 0
    if isinstance(value, list):
        return len(value)
    if isinstance(value, str):
        items = [x.strip() for x in value.split(",") if x.strip()]
        return len(items)
    return 0


def _split_csv_like(value: object) -> list[str]:
    """Split comma-separated Yelp fields into a clean list."""
    if pd.isna(value) or value in (None, "", "None"):
        return []
    if isinstance(value, list):
        return [str(v).strip() for v in value if str(v).strip()]
    return [v.strip() for v in str(value).split(",") if v.strip()]


def _list_length(value: object) -> int:
    """Count items for list-like Yelp string/list fields."""
    return len(_split_csv_like(value))


def filter_reviews_by_years(
    review_df: pd.DataFrame,
    years_range: list[int] | None = None,
    date_col: str = "date",
) -> pd.DataFrame:
    """Filter reviews by an explicit set of calendar years."""
    if not years_range:
        return review_df.copy()

    result = review_df.copy()
    temp_date = pd.to_datetime(result[date_col], errors="coerce")
    target_years = set(years_range)
    result = result[temp_date.dt.year.isin(target_years)]

    return result.copy()


def generate_filtered_reviews(
    review_df: pd.DataFrame,
    years_range: list[int] | None = None,
    date_col: str = "date",
    business_col: str = "business_id",
) -> pd.DataFrame:
    """Apply feature-generation filters; currently only year filtering is enabled."""
    required_review = [date_col, business_col]
    miss_review = [c for c in required_review if c not in review_df.columns]
    if miss_review:
        raise KeyError(f"review_df missing columns: {miss_review}")

    result = filter_reviews_by_years(
        review_df=review_df,
        years_range=years_range,
        date_col=date_col,
    )

    return result.copy()


## Primary Features Block

Main entry point:
- `extract_primary_features(...)`

In [ ]:
def build_review_primary_features(review_df: pd.DataFrame) -> pd.DataFrame:
    """Primary features from review records."""
    required = ["review_id", "user_id", "business_id", "stars", "useful", "funny", "cool", "date"]
    missing = [c for c in required if c not in review_df.columns]
    if missing:
        raise KeyError(f"review_df missing columns: {missing}")
    return review_df.loc[:, required].copy()


def build_user_primary_features(user_df: pd.DataFrame) -> pd.DataFrame:
    """Primary features from user records."""
    required = ["user_id", "review_count", "fans", "friends", "compliment_writer"]
    missing = [c for c in required if c not in user_df.columns]
    if missing:
        raise KeyError(f"user_df missing columns: {missing}")

    out = user_df.loc[:, required].copy()
    out["friends"] = out["friends"].apply(_friends_to_count).astype("Int64")
    out = out.rename(columns={"friends": "friends_count"})
    return out


def extract_primary_features(
    data_dir: str | Path,
    review_df: pd.DataFrame | None = None,
    user_df: pd.DataFrame | None = None,
    nrows: int | None = None,
) -> pd.DataFrame:
    """Main function: extract and merge primary features directly from Yelp raw datasets."""
    data_dir = Path(data_dir)

    if review_df is None:
        review_df = read_jsonl_columns(
            data_dir / REVIEW_FILE,
            ["review_id", "user_id", "business_id", "stars", "useful", "funny", "cool"],
            nrows=nrows,
        )
    if user_df is None:
        user_df = read_jsonl_columns(
            data_dir / USER_FILE,
            ["user_id", "review_count", "fans", "friends", "compliment_writer"],
            nrows=nrows,
        )

    review_features = build_review_primary_features(review_df)
    user_features = build_user_primary_features(user_df)

    merged = review_features.merge(
        user_features,
        on="user_id",
        how="left",
        validate="m:1",
    )
    return merged

## Secondary Features Block

Main entry point:
- `extract_secondary_features(...)`

In [3]:
def _build_reviewer_bias_feature(out: pd.DataFrame) -> pd.Series:
    """Feature: reviewer_bias = stars - user_average_stars (user-level baseline)."""
    return out["stars"] - out["user_average_stars"]


def _build_user_tenure_days_feature(out: pd.DataFrame) -> pd.Series:
    """Feature: user_tenure_days from review date and yelping_since."""
    return (out["date"] - out["yelping_since"]).dt.days.astype("Int64")


def _build_elite_persistence_feature(user_df: pd.DataFrame) -> pd.DataFrame:
    """Feature: elite_persistence from the elite field."""
    user_min = user_df.loc[:, ["user_id", "average_stars", "yelping_since", "elite"]].copy()
    user_min = user_min.rename(columns={"average_stars": "user_average_stars"})
    user_min["yelping_since"] = pd.to_datetime(user_min["yelping_since"], errors="coerce")
    user_min["elite_persistence"] = user_min["elite"].apply(_list_length).astype("Int64")
    return user_min


def _build_review_word_count_feature(out: pd.DataFrame) -> pd.Series:
    """Feature: word count of review text."""
    text_series = out["text"].fillna("").astype(str)
    return text_series.str.split().str.len().astype("Int64")


def _build_review_frequency_feature(review_df: pd.DataFrame) -> pd.DataFrame:
    """Feature: review count per user within the current filtered review span."""
    freq = review_df.groupby("user_id", as_index=False).agg(
        review_frequency=("review_id", "count")
    )
    freq["review_frequency"] = freq["review_frequency"].astype("Int64")
    return freq


def build_secondary_features(
    review_df: pd.DataFrame,
    user_df: pd.DataFrame,
) -> pd.DataFrame:
    """Create level-2 secondary features from merged/derived information."""
    review_required = ["review_id", "user_id", "business_id", "date", "stars", "text"]
    user_required = ["user_id", "average_stars", "yelping_since", "elite"]

    miss_review = [c for c in review_required if c not in review_df.columns]
    miss_user = [c for c in user_required if c not in user_df.columns]
    if miss_review:
        raise KeyError(f"review_df missing columns: {miss_review}")
    if miss_user:
        raise KeyError(f"user_df missing columns: {miss_user}")

    out = review_df.loc[:, ["review_id", "user_id", "business_id", "date", "stars", "text"]].copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce")

    user_min = _build_elite_persistence_feature(user_df)
    out = out.merge(user_min, on="user_id", how="left", validate="m:1")

    review_freq = _build_review_frequency_feature(review_df)
    out = out.merge(review_freq, on="user_id", how="left", validate="m:1")

    out["reviewer_bias"] = _build_reviewer_bias_feature(out)
    out["user_tenure_days"] = _build_user_tenure_days_feature(out)
    out["review_word_count"] = _build_review_word_count_feature(out)

    secondary_cols = [
        "review_id",
        "reviewer_bias",
        "user_tenure_days",
        "elite_persistence",
        "review_word_count",
        "review_frequency",
    ]
    return out.loc[:, secondary_cols]


def extract_secondary_features(
    data_dir: str | Path,
    review_df: pd.DataFrame | None = None,
    user_df: pd.DataFrame | None = None,
    nrows: int | None = None,
) -> pd.DataFrame:
    """Public secondary-feature extractor."""
    data_dir = Path(data_dir)

    if review_df is None:
        review_df = read_jsonl_columns(
            data_dir / REVIEW_FILE,
            [
                "review_id",
                "user_id",
                "business_id",
                "date",
                "stars",
                "text",
                "useful",
                "funny",
                "cool",
            ],
            nrows=nrows,
        )
    if user_df is None:
        user_df = read_jsonl_columns(
            data_dir / USER_FILE,
            [
                "user_id",
                "review_count",
                "fans",
                "friends",
                "compliment_writer",
                "average_stars",
                "yelping_since",
                "elite",
            ],
            nrows=nrows,
        )

    return build_secondary_features(
        review_df=review_df,
        user_df=user_df,
    )

## Features Block

Main entry point:
- `extract_features(...)`

In [4]:
def extract_features(
    data_dir: str | Path,
    review_df: pd.DataFrame | None = None,
    user_df: pd.DataFrame | None = None,
    nrows: int | None = None,
) -> pd.DataFrame:
    """Public feature extractor that joins the primary and secondary feature sets."""
    primary = extract_primary_features(
        data_dir=data_dir,
        review_df=review_df,
        user_df=user_df,
        nrows=nrows,
    )
    secondary = extract_secondary_features(
        data_dir=data_dir,
        review_df=review_df,
        user_df=user_df,
        nrows=nrows,
    )
    return primary.merge(secondary, on="review_id", how="left", validate="1:1")

## Execution: Generate Features CSV

Run the next code cell to load data, apply the year filter, build primary and secondary features, and export the combined output to `processed_data/review_features.csv`.

In [5]:
# Combined feature extraction and export
DATA_DIR = Path("./yelp_dataset")
TEST_NROWS = None #20_000
YEARS_RANGE = [2015, 2016, 2017, 2020, 2021, 2022]

# Load raw data
print("Loading raw data...")
review_sample_for_filter = read_jsonl_columns(
    DATA_DIR / REVIEW_FILE,
    ["review_id", "user_id", "business_id", "date", "stars", "useful", "funny", "cool", "text"],
    nrows=TEST_NROWS,
)
user_data = read_jsonl_columns(
    DATA_DIR / USER_FILE,
    ["user_id", "review_count", "fans", "friends", "compliment_writer", "average_stars", "yelping_since", "elite"],
    nrows=TEST_NROWS,
)

print(f"Raw reviews: {len(review_sample_for_filter)}")

# Apply filters
print("\nApplying filters...")
review_filtered = generate_filtered_reviews(
    review_df=review_sample_for_filter,
    years_range=YEARS_RANGE,
)
print(f"After year filter: {len(review_filtered)}")

# Extract primary features
print("\nExtracting primary features...")
primary_features = extract_primary_features(
    data_dir=DATA_DIR,
    review_df=review_filtered,
    user_df=user_data,
    nrows=TEST_NROWS,
)
print(f"Primary features shape: {primary_features.shape}")

# Extract secondary features
print("Extracting secondary features...")
secondary_features = extract_secondary_features(
    data_dir=DATA_DIR,
    review_df=review_filtered,
    user_df=user_data,
    nrows=TEST_NROWS,
)
print(f"Secondary features shape: {secondary_features.shape}")

# Combine
print("\nCombining features...")
combined_features = primary_features.merge(
    secondary_features,
    on="review_id",
    how="left",
    validate="1:1",
)
print(f"Combined features shape: {combined_features.shape}")

# Save to CSV
print("\nSaving to CSV...")
output_dir = Path("./processed_data")
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "review_features.csv"
combined_features.to_csv(output_path, index=False, float_format="%.3f")
print(f"Saved to: {output_path.resolve()}")

# Summary
print("\n=== SUMMARY ===")
print(f"Total rows: {len(combined_features)}")
print(f"Total columns: {len(combined_features.columns)}")
print(f"\nColumns: {list(combined_features.columns)}")
print(f"\nFirst row preview:")
print(combined_features.iloc[0].to_string())

Loading raw data...
Raw reviews: 6990280

Applying filters...
After year filter: 3471756

Extracting primary features...
Primary features shape: (3471756, 11)
Extracting secondary features...
Secondary features shape: (3471756, 6)

Combining features...
Combined features shape: (3471756, 16)

Saving to CSV...
Saved to: /Users/eukey/Work/docker-share/ubuntu2204-llm-mnt/cai5133-project/processed_data/review_features.csv

=== SUMMARY ===
Total rows: 3471756
Total columns: 16

Columns: ['review_id', 'user_id', 'business_id', 'stars', 'useful', 'funny', 'cool', 'review_count', 'fans', 'friends_count', 'compliment_writer', 'reviewer_bias', 'user_tenure_days', 'elite_persistence', 'review_word_count', 'review_frequency']

First row preview:
review_id            AqPFMleE6RsU23_auESxiA
user_id              _7bHUi9Uuf5__HHc_Q8guQ
business_id          kxX2SOes4o-D3ZQBkiMRfA
stars                                     5
useful                                    1
funny                               